# Real-world Experiments with SSSD

This notebook demonstrates how to run **SSSD (Similar Scores, Similar Data)** on real-world anomaly detection datasets.

A larger SSSD value indicates a stronger violation of score-data consistency and therefore a higher probability of unreliable detection.


## Step 1: Environment setup

The real-world datasets should be placed under the `dataset/realwold` directory. Each `.mat` file is expected to contain:

- `X`: data matrix with shape `(n_samples, n_features)`
- `y`: binary ground-truth labels, where anomalies are labeled as `1`


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat

from sklearn.metrics import roc_auc_score, average_precision_score
from pyod.utils.utility import precision_n_scores, standardizer

from pyod.models.iforest import IForest
from pyod.models.lof import LOF
from pyod.models.ocsvm import OCSVM
from pyod.models.knn import KNN
from pyod.models.hbos import HBOS
from pyod.models.abod import ABOD
from pyod.models.loda import LODA
from pyod.models.pca import PCA

# Make sure SSSD.py in the project root can be imported.
# This works whether the notebook is launched from the project root
# or from the notebooks/ directory.
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
sys.path.insert(0, str(PROJECT_ROOT))

from SSSD import SSSD


# print(f"Project root: {PROJECT_ROOT}")

## Step 2: Configuration

In [2]:
DATA_DIR = PROJECT_ROOT / "dataset"/"realworld"

# You can add or remove datasets depending on which files are available locally.
DATASETS = [
    "glass.mat",
    "ionosphere.mat",
    "letter.mat",
    "lympho.mat",
    "vertebral.mat",
    'vowels.mat',
    "wbc.mat",
]

DETECTORS = [
    ("LODA",LODA()),
    ("ABOD",ABOD()),
    ("iForest",IForest()),
    ('kNN',KNN()),
    ('LOF',LOF()),
    ("HBOS",HBOS()),
    ("OCSVM",OCSVM()),
    ("PCA",PCA())

]

# print(f"Data directory: {DATA_DIR}")

## Step 3: Helper functions

The following helper functions keep the main experiment loop clean.

- `load_odds_mat`: loads an ODDS-style `.mat` dataset.
- `normalize_scores`: maps anomaly scores to `[0, 1]`.
- `evaluate_detector`: trains one detector and computes both external metrics and SSSD.


In [3]:
def load_odds_mat(file_path):
    """Load an ODDS-style .mat anomaly detection dataset."""
    mat = loadmat(file_path)
    X = mat["X"]
    y = mat["y"].ravel().astype(int)

    # Standardization is commonly used for distance-based anomaly detectors.
    X = standardizer(X)
    return X, y


def normalize_scores(scores):
    """Normalize anomaly scores to [0, 1]"""
    scores = np.asarray(scores, dtype=float)
    score_min = np.min(scores)
    score_max = np.max(scores)
    denom = score_max - score_min
    if denom == 0:
        return np.zeros_like(scores, dtype=float)
    return (scores - score_min) / denom


def evaluate_detector(name, detector, X, y, sssd_k=5, sssd_m=5):
    """Train one anomaly detector and compute external metrics plus SSSD."""
    detector.fit(X)

    # PyOD detectors return larger decision_function values for more anomalous samples.
    scores = detector.decision_function(X)
    scores = normalize_scores(scores)

    # External metrics require labels and are used here only for validation/analysis.
    roc_auc = roc_auc_score(y, scores)
    avg_precision = average_precision_score(y, scores)
    precision_at_k = precision_n_scores(y, scores)

    # SSSD is an internal evaluation score computed from X and anomaly scores only.
    sssd = SSSD(X, scores, k=sssd_k, m=sssd_m)
    _, final_sssd_sum, _ = sssd.get_final_SSSD()
    _, score_sssd_sum, _ = sssd.get_Score_SSSD()
    _, distance_sssd_sum, _ = sssd.get_Distance_SSSD()

    return {
        "Detector": name,
        "ROC-AUC": roc_auc,
        "AP": avg_precision,
        "Precision@k": precision_at_k,
        "Final SSSD": final_sssd_sum,
        "Score-SSSD": score_sssd_sum,
        "Distance-SSSD": distance_sssd_sum,
    }

## Step 4: Run real-world experiments

For each available dataset, we train the selected detectors and compute both external metrics and SSSD.

Note that ROC-AUC, AP, and Precision@k are computed only for analysis because they require ground-truth labels. In contrast, SSSD itself does not use labels.


In [4]:
all_results = []

for dataset_name in DATASETS:
    file_path = DATA_DIR / dataset_name
    X, y = load_odds_mat(file_path)

    print(f"Dataset: {dataset_name}")
    print(f"Shape: {X.shape}, anomaly ratio: {np.mean(y):.4f}")

    for detector_name, detector in DETECTORS:
        try:
            result = evaluate_detector(detector_name, detector, X, y)
            result["Dataset"] = dataset_name.replace(".mat", "")
            all_results.append(result)
            print(
                f"  {detector_name:<8} "
                f"ROC-AUC={result['ROC-AUC']:.4f}, "
                f"AP={result['AP']:.4f}, "
                f"Final SSSD={result['Final SSSD']:.4f}"
            )
        except Exception as exc:
            print(f"  {detector_name:<8} failed: {exc}")

results_df = pd.DataFrame(all_results)
results_df

Dataset: glass.mat
Shape: (214, 9), anomaly ratio: 0.0421
  LODA     ROC-AUC=0.4938, AP=0.0541, Final SSSD=2.9501
  ABOD     ROC-AUC=0.7182, AP=0.1113, Final SSSD=3.1220
  iForest  ROC-AUC=0.7024, AP=0.0867, Final SSSD=2.2314
  kNN      ROC-AUC=0.8325, AP=0.1373, Final SSSD=1.5114
  LOF      ROC-AUC=0.8347, AP=0.1462, Final SSSD=2.3614
  HBOS     ROC-AUC=0.7057, AP=0.0742, Final SSSD=3.3954
  OCSVM    ROC-AUC=0.5496, AP=0.1143, Final SSSD=2.5108
  PCA      ROC-AUC=0.5919, AP=0.0699, Final SSSD=2.4609
Dataset: ionosphere.mat
Shape: (351, 33), anomaly ratio: 0.3590
  LODA     ROC-AUC=0.7467, AP=0.6937, Final SSSD=9.8379
  ABOD     ROC-AUC=0.9170, AP=0.8973, Final SSSD=5.7879
  iForest  ROC-AUC=0.8611, AP=0.8173, Final SSSD=6.4783
  kNN      ROC-AUC=0.9337, AP=0.9372, Final SSSD=3.3877
  LOF      ROC-AUC=0.8939, AP=0.8391, Final SSSD=5.6125
  HBOS     ROC-AUC=0.5661, AP=0.3629, Final SSSD=10.2435
  OCSVM    ROC-AUC=0.8510, AP=0.8391, Final SSSD=6.9139
  PCA      ROC-AUC=0.7944, AP=0.7471,

,Detector,ROC-AUC,AP,Precision@k,Final SSSD,Score-SSSD,Distance-SSSD,Dataset
0,LODA,0.493767,0.054122,0.000000,2.950066,2.884341,3.015792,glass
1,ABOD,0.718157,0.111295,0.111111,3.122021,4.142693,2.101349,glass
2,iForest,0.702439,0.086680,0.111111,2.231411,2.999566,1.463257,glass
3,kNN,0.832520,0.137275,0.111111,1.511359,2.200694,0.822023,glass
4,LOF,0.834688,0.146175,0.111111,2.361397,2.076831,2.645963,glass
5,HBOS,0.705691,0.074214,0.000000,3.395413,4.001380,2.789446,glass
6,OCSVM,0.549593,0.114303,0.111111,2.510755,2.922086,2.099425,glass
7,PCA,0.591870,0.069894,0.111111,2.460940,2.477045,2.444836,glass
8,LODA,0.746667,0.693663,0.579365,9.837938,9.713199,9.962677,ionosphere
9,ABOD,0.916966,0.897291,0.801587,5.787944,5.297935,6.277952,ionosphere


## Step 5: Summarize detector-level results

The table below provides a compact summary of all available datasets, with values sorted in descending order of AP.

In [5]:
if len(results_df) > 0:
    summary_df = (
        results_df
        .groupby("Detector")[["ROC-AUC", "AP", "Precision@k", "Final SSSD", "Score-SSSD", "Distance-SSSD"]]
        .mean()
        .sort_values("AP", ascending=False)
    )
    display(summary_df.round(4))
else:
    print("No results to summarize. Please add datasets under the data/ directory and rerun the notebook.")

,ROC-AUC,AP,Precision@k,Final SSSD,Score-SSSD,Distance-SSSD
Detector,,,,,,
kNN,0.8452,0.4743,0.4542,6.2389,8.9174,3.5603
LOF,0.8460,0.4441,0.4273,8.1786,5.8500,10.5072
ABOD,0.7968,0.4058,0.4002,7.6838,6.9233,8.4443
iForest,0.7443,0.3987,0.3520,10.5518,9.6601,11.4434
OCSVM,0.7310,0.3821,0.3522,11.0378,10.2999,11.7757
PCA,0.6882,0.3496,0.3103,10.8168,9.0252,12.6084
HBOS,0.6821,0.3250,0.2918,11.7505,10.6178,12.8831
LODA,0.6604,0.2919,0.2563,11.5537,9.6841,13.4233
